# A/B Testing & Causal Inference

Running valid experiments, avoiding common traps (peeking, SUTVA violations),
and measuring causal effects rather than correlations.

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "figure.dpi":        130,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
    "font.family":       "sans-serif",
    "lines.linewidth":   2.2,
    "patch.edgecolor":   "none",
})
C = ["#2563EB","#DC2626","#16A34A","#D97706","#7C3AED","#0891B2","#DB2777"]
print("Setup complete")

## 1. The peeking problem

Checking results before your pre-specified sample size and stopping at
first significance inflates the false-positive rate from 5% to ~30%.

In [ ]:
from scipy import stats

rng    = np.random.default_rng(42)
n_days, n_per_day, n_sims, alpha = 30, 100, 3000, 0.05

fp_fixed, fp_peeking = 0, 0
peeking_stop_days = []

for _ in range(n_sims):
    ctrl, trt = [], []
    found = False
    for day in range(n_days):
        ctrl += rng.binomial(1, 0.10, n_per_day).tolist()
        trt  += rng.binomial(1, 0.10, n_per_day).tolist()
        if len(ctrl) >= 40:
            _, p = stats.ttest_ind(ctrl, trt)
            if p < alpha and not found:
                fp_peeking += 1
                peeking_stop_days.append(day + 1)
                found = True
    _, p = stats.ttest_ind(ctrl, trt)
    if p < alpha:
        fp_fixed += 1

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
strategies = ["Fixed horizon
(alpha=5%)", "Peeking
(check daily)"]
fp_rates   = [fp_fixed/n_sims, fp_peeking/n_sims]
bars = ax.bar(strategies, [r*100 for r in fp_rates], color=[C[2], C[1]], width=0.4)
ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=12, fontweight="bold")
ax.axhline(5, color="grey", lw=1.5, linestyle="--", label="Nominal alpha=5%")
ax.set(ylabel="False positive rate (%)", ylim=(0, 45),
       title=f"FPR in A/A tests
({n_sims:,} simulations, 30 days, 100 obs/day)")
ax.legend()

ax = axes[1]
if peeking_stop_days:
    ax.hist(peeking_stop_days, bins=range(1,32), color=C[1], alpha=0.85,
            edgecolor="white", linewidth=0.4)
ax.set(xlabel="Day of early stop", ylabel="Count",
       title="When does peeking falsely 'find' significance?
(mostly in early days when variance is high)")
plt.tight_layout()
plt.show()
print(f"Fixed-horizon FPR: {fp_fixed/n_sims:.1%}  (nominal = 5%)")
print(f"Peeking FPR:       {fp_peeking/n_sims:.1%}  (inflated by {fp_peeking/fp_fixed:.1f}x)")

## 2. CUPED — variance reduction with pre-experiment data

Regress out a pre-experiment covariate correlated with the post-experiment
metric. Equivalent sample size reduction shown below.

In [ ]:
from scipy import stats

rng = np.random.default_rng(42)
n   = 2000

pre   = rng.normal(100, 20, n)
treat = rng.binomial(1, 0.5, n)
post  = 0.8*pre + treat*3 + rng.normal(0, 12, n)

theta      = np.cov(post, pre)[0,1] / np.var(pre)
post_cuped = post - theta*(pre - pre.mean())

var_reduction = 1 - np.var(post_cuped) / np.var(post)

# Standard vs CUPED estimate
t_std, p_std = stats.ttest_ind(post[treat==1], post[treat==0])
t_cup, p_cup = stats.ttest_ind(post_cuped[treat==1], post_cuped[treat==0])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Scatter: pre vs post (correlation)
ax = axes[0]
for t, col, label in [(0, C[0], "Control"), (1, C[1], "Treatment")]:
    ax.scatter(pre[treat==t], post[treat==t], alpha=0.2, s=10, color=col, label=label)
ax.set(xlabel="Pre-experiment metric", ylabel="Post-experiment metric",
       title=f"Pre-post correlation
(r = {np.corrcoef(pre, post)[0,1]:.2f})")
ax.legend(markerscale=2, fontsize=9)

# Distributions before and after CUPED
ax = axes[1]
for data, name, col, lw in [(post, "Raw", C[0], 1.5), (post_cuped, "CUPED", C[2], 2.5)]:
    ctrl_vals = data[treat==0]; trt_vals = data[treat==1]
    ax.hist(ctrl_vals - ctrl_vals.mean(), bins=40, alpha=0.35, color=col, density=True)
    ax.hist(trt_vals  - trt_vals.mean(),  bins=40, alpha=0.35, color=col, density=True)
ax.set(xlabel="Demeaned metric", ylabel="Density",
       title=f"Variance reduction: {var_reduction:.0%}
(CUPED = narrower = more power)")

# p-value comparison
ax = axes[2]
metrics   = ["Standard t-test", "CUPED t-test"]
p_values  = [p_std, p_cup]
bar_colors = [C[0] if p>0.05 else C[2] for p in p_values]
bars = ax.bar(metrics, [-np.log10(p) for p in p_values], color=bar_colors, width=0.4)
ax.axhline(-np.log10(0.05), color=C[1], lw=1.8, linestyle="--",
           label="p=0.05 threshold")
ax.bar_label(bars, labels=[f"p={p:.4f}" for p in p_values], padding=4, fontsize=10)
ax.set(ylabel="-log10(p-value)  (higher = more significant)",
       title="CUPED achieves same significance
with fewer samples")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Variance reduction:  {var_reduction:.1%}")
print(f"Equivalent to:       {1/(1-var_reduction):.1f}x more data")

## 3. Heterogeneous treatment effects (T-learner)

Not everyone responds equally to a treatment. The T-learner estimates
per-user CATE by training separate response surface models and subtracting.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

rng = np.random.default_rng(42)
n   = 4000

age   = rng.uniform(20, 70, n)
spend = rng.lognormal(4, 0.8, n)
p_t   = 1 / (1 + np.exp(-(age - 45) / 10))
treat = rng.binomial(1, p_t, n)
true_cate = 4 + 0.08*(60 - age) + 0.0008*spend
outcome   = 15 + 0.05*age + 0.003*spend + true_cate*treat + rng.normal(0, 4, n)

X = np.column_stack([age, spend])
gbm1 = GradientBoostingRegressor(100, random_state=42).fit(X[treat==1], outcome[treat==1])
gbm0 = GradientBoostingRegressor(100, random_state=42).fit(X[treat==0], outcome[treat==0])
cate_hat = gbm1.predict(X) - gbm0.predict(X)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# True vs estimated CATE
ax = axes[0]
ax.scatter(true_cate, cate_hat, alpha=0.15, s=8, color=C[0])
lo, hi = min(true_cate.min(), cate_hat.min()), max(true_cate.max(), cate_hat.max())
ax.plot([lo,hi],[lo,hi], "--", color=C[1], lw=2, label=f"r={np.corrcoef(true_cate,cate_hat)[0,1]:.2f}")
ax.set(xlabel="True CATE", ylabel="Estimated CATE",
       title="T-learner: true vs estimated CATE
(Pearson r shown)")
ax.legend()

# CATE vs age
ax = axes[1]
order = np.argsort(age)
ax.scatter(age, true_cate, alpha=0.15, s=8, color=C[0], label="True CATE")
ax.scatter(age, cate_hat,  alpha=0.15, s=8, color=C[1], label="Estimated CATE")
ax.set(xlabel="Age", ylabel="CATE", title="CATE heterogeneity by age
(younger users respond more)")
ax.legend(markerscale=2, fontsize=9)

# Cumulative gain: target high-CATE users
ax = axes[2]
order = np.argsort(-cate_hat)
cumulative_true = np.cumsum(true_cate[order]) / true_cate.sum()
cumulative_rand = np.linspace(0, 1, n)
ax.plot(np.linspace(0,100,n), cumulative_true*100, color=C[0], lw=2, label="CATE targeting")
ax.plot([0,100],[0,100], "--", color="grey", lw=1.5, label="Random targeting")
ax.fill_between(np.linspace(0,100,n), cumulative_true*100, np.linspace(0,100,n),
                alpha=0.15, color=C[0])
ax.set(xlabel="% customers targeted", ylabel="% cumulative uplift captured",
       title="Cumulative gain curve
(CATE targeting outperforms random)")
ax.legend()
plt.tight_layout()
plt.show()
top20_gain = cumulative_true[int(0.2*n)] * 100
print(f"Targeting top 20% by CATE captures {top20_gain:.0f}% of total uplift")
print(f"vs {20:.0f}% if targeting randomly")